In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
## 1. tutorial of median filter

In [ ]:
def nan_median_filter(image, kernel_size=3, iterations=1):
    """
    Apply a median filter while ignoring NaN values.
    Pixels that are originally NaN remain NaN after filtering.
    
    Parameters:
        image (np.ndarray): Input image as a 2D NumPy array with NaN background.
        kernel_size (int): Odd-sized filter window, such as 3, 5, or 7.
        iterations (int): Number of filtering iterations.
        
    Returns:
        np.ndarray: Median-filtered image.
    """
    if kernel_size % 2 == 0:
        raise ValueError("kernel_size must be an odd number")
    
    # Store the original NaN positions
    nan_mask = np.isnan(image)
    
    filtered = image.copy()
    pad = kernel_size // 2
    
    for _ in range(iterations):
        # Pad with NaN so only valid values are included near the boundaries
        padded = np.pad(filtered, pad_width=pad, mode='constant', constant_values=np.nan)
        new_image = np.empty_like(filtered)
        
        # Compute the median of each window while ignoring NaN values
        for i in range(filtered.shape[0]):
            for j in range(filtered.shape[1]):
                window = padded[i:i+kernel_size, j:j+kernel_size]
                new_image[i, j] = np.nanmedian(window)
        
        # Restore the original NaN positions
        new_image[nan_mask] = np.nan
        filtered = new_image
    
    return filtered

In [ ]:
## kernel_size=3 
import itertools

adata = adata_sub.copy()
cell_type_list = ['Cancer_Epithelial', 'Endothelial']

for cell_type in cell_type_list:
    df = sc.get.obs_df(adata, keys = [cell_type, 'array_col', 'array_row'])
    
    col_numbers = numbers = list(range(int(df['array_col'].describe().iloc[3]), int(df['array_col'].describe().iloc[7])))
    row_numbers = numbers = list(range(int(df['array_row'].describe().iloc[3]), int(df['array_row'].describe().iloc[7])))
    pairs = list(itertools.product(col_numbers, row_numbers))
    df_pairs = pd.DataFrame(pairs, columns=['array_col', 'array_row'])
    
    merged_df = pd.merge(df, df_pairs, how='outer')
    mat = merged_df.pivot(index='array_row', columns='array_col', values=cell_type)
    img = mat.to_numpy()
    img_filter = nan_median_filter(img, kernel_size=3, iterations=1)
    img_filter = pd.DataFrame(img_filter, index = mat.index, columns = mat.columns)
    df_melt = img_filter.reset_index().melt(id_vars='array_row',
                                            var_name='array_col',
                                            value_name=cell_type)
    df_melt.columns = ['array_row', 'array_col', f'{cell_type}_filtered']
    df_final = pd.merge(df, df_melt, how='inner')
    df_final.index = df.index.copy()
    adata.obs[f'{cell_type}_filtered'] = df_final[f'{cell_type}_filtered'].copy()

sc.settings.set_figure_params(dpi_save=300, facecolor='white')

for cell_type in cell_type_list:
    fig, axs = plt.subplots(1, 2, figsize=(18, 9))

    sc.pl.spatial(adata, color=f'{cell_type}', ax=axs[0], show=False, frameon = False)
    axs[0].set_title(f'{cell_type}', fontsize = 20)

    sc.pl.spatial(adata, color=f'{cell_type}_filtered', ax=axs[1], show=False, frameon = False)
    axs[1].set_title(f'{cell_type}_filtered', fontsize = 20)

    plt.tight_layout()
    plt.show()

In [ ]:
for cell_type in cell_type_list:
    fig, axs = plt.subplots(1, 2, figsize=(18, 9))

    sc.pl.spatial(adata, color=f'{cell_type}', ax=axs[0], show=False, frameon = False)
    axs[0].set_title(f'{cell_type}', fontsize = 40)

    sc.pl.spatial(adata, color=f'{cell_type}_filtered', ax=axs[1], show=False, frameon = False)
    axs[1].set_title(f'{cell_type}_filtered', fontsize = 40)

    plt.tight_layout()
    plt.show()

In [ ]:
## 2. All data processing ## 
import os

directory_path = '../data/h5ad_segmented'
all_files = os.listdir(directory_path)

files_list = [
    f.replace('_segmented.h5ad', '')  
    for f in all_files
    if '_segmented.h5ad' in f  
]

files_list.sort()

In [ ]:
for file in files_list:
    adata = sc.read_h5ad(f'../data/h5ad_segmented/{file}_segmented.h5ad')
    cell_type_list = adata.obsm['tacco_annotation'].columns.tolist()
    for cell_type in cell_type_list:
        df = sc.get.obs_df(adata, keys = [cell_type, 'array_col', 'array_row'])
    
        col_numbers = numbers = list(range(int(df['array_col'].describe().iloc[3]), int(df['array_col'].describe().iloc[7])))
        row_numbers = numbers = list(range(int(df['array_row'].describe().iloc[3]), int(df['array_row'].describe().iloc[7])))
        pairs = list(itertools.product(col_numbers, row_numbers))
        df_pairs = pd.DataFrame(pairs, columns=['array_col', 'array_row'])
    
        merged_df = pd.merge(df, df_pairs, how='outer')
        mat = merged_df.pivot(index='array_row', columns='array_col', values=cell_type)
        img = mat.to_numpy()
        img_filter = nan_median_filter(img, kernel_size=3, iterations=1)
        img_filter = pd.DataFrame(img_filter, index = mat.index, columns = mat.columns)
        df_melt = img_filter.reset_index().melt(id_vars='array_row',
                                                var_name='array_col',
                                                value_name=cell_type)
        df_melt.columns = ['array_row', 'array_col', f'{cell_type}_filtered']
        df_final = pd.merge(df, df_melt, how='inner')
        df_final.index = df.index.copy()
        adata.obs[f'{cell_type}_filtered'] = df_final[f'{cell_type}_filtered'].copy()

    adata.write_h5ad(f'../data/h5ad_segmented_mf/{file}_mf.h5ad')